# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FrinceNacion/Starter-Notebooks/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Ranking / scoring. The decision is which content pages editors should refresh first, so the model should produce a ranked opportunity score for each page rather than a binary prediction or an unsupervised grouping.


In [14]:
from pathlib import Path
import pandas as pd

repo_root = Path.cwd().parents[1]
csv_path = repo_root / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(csv_path)
print(f"Loaded {len(df)} rows with {len(df.columns)} columns")
print("Sample columns:", df.columns[:10].tolist())

Loaded 30000 rows with 44 columns
Sample columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count']


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

We would predict a page-level refresh opportunity score. The score will be based on observed page performance and trend outcomes in the data — for example, traffic, position, engagement, and direction of trend — rather than a hand-defined binary rule.

In [15]:
print("Examples of observed signal columns that support a ranking target:")
print(df[["sessions_last_30d", "sessions_prev_30d", "trend_pct", "avg_position", "search_volume"]].head(5))

Examples of observed signal columns that support a ranking target:
   sessions_last_30d  sessions_prev_30d  trend_pct  avg_position  \
0                  2                  9      -41.4          10.6   
1                  3                  2      -57.7          20.3   
2                  1                  3      -60.9          36.5   
3                 35                 26      -13.8           6.2   
4                 14                  9      -34.7          44.0   

   search_volume  
0           10.0  
1           90.0  
2            0.0  
3           10.0  
4            0.0  


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@10 on the ranked refresh queue. Good means the top 10 recommended pages include at least 70% of the highest-opportunity pages by the observed performance/trend proxy, which would concentrate editorial effort on the strongest refresh candidates.

In [16]:
proxy_score = (df["search_volume"] * (1 - df["avg_position"] / 100) * (1 + (-df["trend_pct"]).clip(lower=0) / 100)).fillna(0)
df = df.assign(proxy_score=proxy_score)
print("Proxy score summary:")
print(df["proxy_score"].describe())
print("Top 5 pages by score:")
print(df.sort_values("proxy_score", ascending=False).head(5)[["content_id", "proxy_score", "search_volume", "avg_position", "trend_pct"]])

Proxy score summary:
count    30000.000000
mean       134.057342
std       1314.109888
min        -14.400000
25%          0.000000
50%          0.000000
75%         18.942862
max      61775.752500
Name: proxy_score, dtype: float64
Top 5 pages by score:
                 content_id  proxy_score  search_volume  avg_position  \
13502  content_f76ccf7a7834   61775.7525        49500.0           9.5   
28282  content_454cc6654c6e   50936.6440        60500.0          44.9   
22788  content_ee4630879d03   47874.6180        49500.0          25.2   
12140  content_ef99c4abd9ab   45510.0000        74000.0          38.5   
9217   content_12e48d4b449d   44986.0000        27100.0          17.0   

       trend_pct  
13502      -37.9  
28282      -52.8  
22788      -29.3  
12140        3.6  
9217      -100.0  


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content page / refresh candidate. Each row contains page-level performance signals and content metadata used to compute a ranked opportunity score.

In [17]:
print("Unit of analysis: one row per content page")
print(df.loc[:, ["content_id", "search_volume", "avg_position", "trend_pct", "sessions_last_30d", "content_age_days"]].head(5))

Unit of analysis: one row per content page
             content_id  search_volume  avg_position  trend_pct  \
0  content_304f48230142           10.0          10.6      -41.4   
1  content_a1fb4e703a9e           90.0          20.3      -57.7   
2  content_9aa793d4d895            0.0          36.5      -60.9   
3  content_331d6c4de07b           10.0           6.2      -13.8   
4  content_d99b7a2d90ca            0.0          44.0      -34.7   

   sessions_last_30d  content_age_days  
0                  2               187  
1                  3               445  
2                  1               141  
3                 35               463  
4                 14               263  


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Opportunity depends on a mix of search volume, position, engagement, freshness, and trend direction. A hard rule cannot easily balance these competing signals or rank pages by overall value, so ML is needed to learn the best combination of signals for the refresh queue.

In [18]:
correlations = df[["search_volume", "avg_position", "trend_pct", "engagement_rate", "sessions_last_30d"]].corr()
print("Key signal correlations:")
print(correlations)

Key signal correlations:
                   search_volume  avg_position  trend_pct  engagement_rate  \
search_volume           1.000000      0.045354   0.002515        -0.010146   
avg_position            0.045354      1.000000   0.047248        -0.017926   
trend_pct               0.002515      0.047248   1.000000         0.008412   
engagement_rate        -0.010146     -0.017926   0.008412         1.000000   
sessions_last_30d      -0.010160     -0.038176   0.007872         0.004127   

                   sessions_last_30d  
search_volume              -0.010160  
avg_position               -0.038176  
trend_pct                   0.007872  
engagement_rate             0.004127  
sessions_last_30d           1.000000  


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.